# TelAgentBench-ID Localization Pipeline
### 🇮🇩 Adaptasi Domain Telko Korea ke Telekomunikasi Indonesia (*TelkomNusa, IndoSatria, XLAxiata*)

Notebook ini sudah diperbaiki untuk mendukung format **Standard JSON Array** maupun **JSON-Lines (`.jsonl`)** secara otomatis:
- **Universal JSON/JSONL Parser**: Menangani file seperti `250918_telcoFC_tools.json` dan multi-line Action files tanpa `JSONDecodeError Extra data`.
- ⏩ **Instant Resume Active**: `validation.json` yang sudah selesai 100% akan langsung di-skip otomatis (tidak perlu menunggu 38 menit lagi!).
- **High Throughput Batching**: Batch size 15-20 untuk kecepatan maksimal via `gpt-5-mini`.
- **Strict AST & JSON Integrity**: Menjaga 100% struktur fungsi API, ID, dan parameter.
- **Export Archive**: Mengompres hasil ke `dataset_telagentbench_indonesia.zip` siap unduh.

## Section 0: Setup & Install Modern SDKs

In [ ]:
!pip install -q --no-warn-conflicts openai deep-translator requests tqdm

In [ ]:
import os
import re
import json
import time
import copy
import shutil
from pathlib import Path
from typing import List, Dict, Any, Optional, Union, Tuple
from tqdm import tqdm

print(" Environment & Universal Parsers Siap!")

## Section 1: Konfigurasi Provider Aktif (Bluesminds GPT-5-Mini & OpenRouter)

In [ ]:
# DAFTAR PROVIDER API AKTIF & SUPER CEPAT

OPENROUTER_KEY = "YOUR_OPENROUTER_API_KEY_HERE"
BLUESMINDS_KEY = "YOUR_API_KEY_HERE"

CONFIG_PROVIDERS = [
    # 1. Bluesminds - GPT-5-Mini (Primary LLM endpoint)
    {
        "name": "Bluesminds-GPT5-Mini",
        "type": "openai",
        "api_key": BLUESMINDS_KEY,
        "base_url": "https://api.bluesminds.com/v1",
        "model": "gpt-5-mini",
        "rpm_delay": 0.2
    },
    # 2. OpenRouter - Free Auto Models
    {
        "name": "OpenRouter-Free",
        "type": "openrouter",
        "api_key": OPENROUTER_KEY,
        "base_url": "https://openrouter.ai/api/v1",
        "model": "google/gemma-2-9b-it:free",
        "rpm_delay": 0.5
    }
]

print(" Menguji koneksi provider utama...\n")
from openai import OpenAI
from deep_translator import GoogleTranslator

VALID_PROVIDERS = []
for p in CONFIG_PROVIDERS:
    try:
        headers = {}
        if p["type"] == "openrouter":
            headers = {"HTTP-Referer": "https://kaggle.com", "X-Title": "TelAgentBench Translator"}
        client = OpenAI(api_key=p["api_key"], base_url=p["base_url"], default_headers=headers)
        resp = client.chat.completions.create(
            model=p["model"],
            messages=[{"role": "user", "content": "OK"}],
            max_tokens=5
        )
        if resp.choices:
            print(f"[{p['name']}] KONEKSI SUKSES! (Model: {p['model']})")
            p["client"] = client
            VALID_PROVIDERS.append(p)
    except Exception as e:
        print(f"[{p['name']}] Skip: {str(e)[:80]}...")

print(f"\n {len(VALID_PROVIDERS)} Cloud AI Provider Siap + 1 Auto-Fallback DeepTranslator (with fallback)!")

## Section 2: Industrial Universal Translator Engine

In [ ]:
class MultiProviderTranslator:
    def __init__(self, providers: List[Dict[str, Any]]):
        self.providers = providers
        self.current_idx = 0
        from deep_translator import GoogleTranslator
        self.backup_engine = GoogleTranslator(source='ko', target='id')

        self.system_prompt = """Anda adalah penerjemah dan pakar lokalisasi domain telekomunikasi profesional Indonesia.
Tugas Anda: Menerjemahkan kalimat/query telekomunikasi dari BAHASA KOREA ke BAHASA INDONESIA yang alami, resmi, dan sesuai konteks layanan pelanggan.

ATURAN LOKALISASI ENTITAS:
1. Operator: SKT / SK Telecom -> TelkomNusa, KT -> IndoSatria, LGU+ -> XLAxiata, T world -> MyTelkomNusa
2. Paket: baro roaming -> Paket Roaming Passport, 리필쿠폰 -> Kupon Extra Kuota, 부가서비스 -> Layanan Tambahan (Add-on)
3. Layanan: 실시간 요금 -> Tagihan Berjalan, 확정 요금 -> Tagihan Bulanan Resmi, 소액 결제 -> Layanan Potong Pulsa, 미납 요금 -> Tunggakan Tagihan
4. Mata Uang: 원 (KRW) -> IDR / Rp (misal 10.000 won -> Rp100.000), 서울 (Seoul) -> Jakarta

PENTING:
- Kembalikan HANYA JSON list of string ["terjemahan_1", "terjemahan_2", ...] dengan jumlah item PERSIS SAMA dengan input!"""

    def apply_regex_localization(self, text: str) -> str:
        replacements = [
            (r'\bSKT\b|SK텔레콤|SK Telecom', 'TelkomNusa'),
            (r'\bKT\b|케이티', 'IndoSatria'),
            (r'\bLGU\+?\b|LG유플러스|LG U\+', 'XLAxiata'),
            (r'T world|T월드|T membership|T멤버십', 'MyTelkomNusa'),
            (r'baro roaming|baro 로밍|바로그로밍|바로운', 'Paket Roaming Passport'),
            (r'리필쿠폰', 'Kupon Extra Kuota'),
            (r'부가서비스', 'Layanan Tambahan'),
            (r'실시간 요금', 'Tagihan Berjalan'),
            (r'확정 요금', 'Tagihan Bulanan Resmi'),
            (r'소액 결제', 'Potong Pulsa'),
            (r'미납 요금', 'Tunggakan Tagihan'),
            (r'잔여 약정', 'Sisa Masa Kontrak'),
            (r'가족 데이터', 'Kuota Keluarga'),
            (r'서울', 'Jakarta'),
            (r'KRW|원(?=[\s\.,\)]|$)', ' IDR')
        ]
        for pat, repl in replacements:
            text = re.sub(pat, repl, text, flags=re.IGNORECASE)
        return text

    def get_next_provider(self) -> Optional[Dict[str, Any]]:
        if not self.providers:
            return None
        p = self.providers[self.current_idx]
        self.current_idx = (self.current_idx + 1) % len(self.providers)
        return p

    def translate_batch(self, texts: List[str]) -> List[str]:
        if not texts:
            return []
        
        has_hangul = any(bool(re.search(r'[\uac00-\ud7af]', t)) for t in texts)
        if not has_hangul:
            return [self.apply_regex_localization(t) for t in texts]

        if self.providers:
            for _ in range(len(self.providers) * 2):
                provider = self.get_next_provider()
                client = provider.get("client")
                user_content = f"Terjemahkan daftar teks bahasa Korea berikut ke Bahasa Indonesia domain Telko:\n{json.dumps(texts, ensure_ascii=False, indent=2)}"
                
                try:
                    resp = client.chat.completions.create(
                        model=provider["model"],
                        messages=[{"role": "user", "content": f"{self.system_prompt}\n\n{user_content}"}],
                        temperature=0.1,
                        timeout=30
                    )
                    out_text = resp.choices[0].message.content.strip()
                    if out_text.startswith("```"):
                        out_text = re.sub(r"^```(?:json)?|```$", "", out_text, flags=re.MULTILINE).strip()

                    parsed = json.loads(out_text)
                    if isinstance(parsed, dict):
                        for k, v in parsed.items():
                            if isinstance(v, list):
                                parsed = v
                                break
                    
                    if isinstance(parsed, list) and len(parsed) == len(texts):
                        time.sleep(provider.get("rpm_delay", 0.2))
                        return [self.apply_regex_localization(str(x)) for x in parsed]
                except Exception:
                    continue

        # Fallback Otomatis
        fallback_results = []
        for t in texts:
            if re.search(r'[\uac00-\ud7af]', t):
                try:
                    trans = self.backup_engine.translate(t)
                    fallback_results.append(self.apply_regex_localization(trans))
                except:
                    fallback_results.append(self.apply_regex_localization(t))
            else:
                fallback_results.append(self.apply_regex_localization(t))
        return fallback_results

print(" MultiProviderTranslator Engine Siap!")

## Section 3: Universal JSON / JSONL Loader & Automatic Path Resolver

In [ ]:
def load_universal_json(file_path: str) -> Tuple[Any, bool]:
    """
    Membaca file JSON standar maupun JSON-Lines (.jsonl yang berekstensi .json).
    Mengembalikan tuple: (data, is_jsonl)
    """
    with open(file_path, "r", encoding="utf-8") as f:
        raw = f.read().strip()
    if not raw:
        return [], False

    try:
        data = json.loads(raw)
        return data, False
    except json.JSONDecodeError:
        # File berformat JSON-Lines (satu JSON object per baris)
        lines = [l.strip() for l in raw.split("\n") if l.strip()]
        data = [json.loads(l) for l in lines]
        return data, True

def save_universal_json(data: Any, out_path: str, is_jsonl: bool):
    """
    Menyimpan data kembali ke format aslinya (JSON array / JSON lines).
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as out_f:
        if is_jsonl and isinstance(data, list):
            for item in data:
                out_f.write(json.dumps(item, ensure_ascii=False) + "\n")
        else:
            json.dump(data, out_f, ensure_ascii=False, indent=2)

# Deteksi Path Input & Output
CANDIDATE_INPUTS = [
    "/kaggle/input/datasets/mrislantristansyah/tellagentbench/dataset_telagentbench_korea",
    "/kaggle/input/tellagentbench/dataset_telagentbench_korea",
    "/kaggle/input/dataset_telagentbench_korea",
    "dataset_telagentbench_korea",
    "Skripsi SLm/dataset_telagentbench_korea"
]

INPUT_ROOT = None
for path in CANDIDATE_INPUTS:
    if os.path.exists(path):
        INPUT_ROOT = path
        break

if not INPUT_ROOT and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "dataset_telagentbench_korea" in dirs:
            INPUT_ROOT = os.path.join(root, "dataset_telagentbench_korea")
            break
        elif "TelAgent_Action" in dirs:
            INPUT_ROOT = root
            break

if not INPUT_ROOT:
    INPUT_ROOT = "/kaggle/input/datasets/mrislantristansyah/tellagentbench/dataset_telagentbench_korea"

if os.path.exists("/kaggle/working"):
    OUTPUT_ROOT = "/kaggle/working/dataset_telagentbench_indonesia"
else:
    OUTPUT_ROOT = "dataset_telagentbench_indonesia"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f" [INPUT DATASET] : {INPUT_ROOT}")
print(f" [OUTPUT DATASET]: {OUTPUT_ROOT}")

In [ ]:
def process_json_file(file_path: str, out_path: str, translator: MultiProviderTranslator, batch_size: int = 15):
    """
    Menerjemahkan file JSON/JSONL TelAgentBench secara modular, menjaga key dan AST structure 100% utuh.
    """
    # Fitur Resume: Jika file output sudah ada dan valid, lewati langsung
    if os.path.exists(out_path):
        try:
            check_data, _ = load_universal_json(out_path)
            if check_data:
                print(f"⏩ [RESUME/SKIP] File sudah selesai: {os.path.basename(out_path)}")
                return
        except:
            pass

    data, is_jsonl = load_universal_json(file_path)
    if not data:
        return

    translated_data = copy.deepcopy(data)
    text_items = []

    # KASUS A: List of items (Action queries, Plan, IF, Tools)
    if isinstance(translated_data, list):
        for idx, item in enumerate(translated_data):
            if isinstance(item, dict):
                # 1. Action queries
                if "question" in item and isinstance(item["question"], list):
                    for q_group_idx, q_group in enumerate(item["question"]):
                        if isinstance(q_group, list):
                            for msg_idx, msg in enumerate(q_group):
                                if isinstance(msg, dict) and "content" in msg:
                                    text_items.append((idx, ("question", q_group_idx, msg_idx, "content"), msg["content"]))
                
                # 2. Plan query
                if "query" in item and isinstance(item["query"], str):
                    text_items.append((idx, ("query",), item["query"]))
                
                # 3. IF turn prompts
                for turn_key in ["turn_1", "turn_2", "turn_3"]:
                    if turn_key in item and isinstance(item[turn_key], dict) and "prompt" in item[turn_key]:
                        text_items.append((idx, (turn_key, "prompt"), item[turn_key]["prompt"]))
                
                # 4. Tools description (250918_telcoFC_tools.json)
                if "description" in item and isinstance(item["description"], str):
                    text_items.append((idx, ("description",), item["description"]))
                if "parameters" in item and isinstance(item["parameters"], dict):
                    props = item["parameters"].get("properties", {})
                    if isinstance(props, dict):
                        for prop_name, prop_val in props.items():
                            if isinstance(prop_val, dict) and "description" in prop_val:
                                text_items.append((idx, ("parameters", "properties", prop_name, "description"), prop_val["description"]))

    if text_items:
        print(f"[*] Menerjemahkan {len(text_items)} teks di {os.path.basename(file_path)}...")
        for i in tqdm(range(0, len(text_items), batch_size), desc=f"Translating {os.path.basename(file_path)}"):
            batch = text_items[i:i+batch_size]
            raw_texts = [b[2] for b in batch]
            translated_texts = translator.translate_batch(raw_texts)
            
            for b_idx, trans_text in enumerate(translated_texts):
                item_idx, field_path, _ = batch[b_idx]
                target_obj = translated_data[item_idx]
                for step in field_path[:-1]:
                    target_obj = target_obj[step]
                target_obj[field_path[-1]] = trans_text

    save_universal_json(translated_data, out_path, is_jsonl)
    print(f" Sukses Disimpan: {out_path}")

## Section 4: Eksekusi Penerjemahan Seluruh Dataset TelAgentBench

In [ ]:
translator_engine = MultiProviderTranslator(VALID_PROVIDERS)

# 1. Salin folder possible_answer (Ground Truth AST) secara utuh
src_possible = os.path.join(INPUT_ROOT, "TelAgent_Action", "possible_answer")
dst_possible = os.path.join(OUTPUT_ROOT, "TelAgent_Action", "possible_answer")
if os.path.exists(src_possible):
    os.makedirs(dst_possible, exist_ok=True)
    for f_name in os.listdir(src_possible):
        shutil.copy2(os.path.join(src_possible, f_name), os.path.join(dst_possible, f_name))
    print(f" Salinan Ground Truth possible_answer berhasil disinkronkan!")

# 2. Cari seluruh file JSON di dataset input
all_json_files = []
for root, dirs, files in os.walk(INPUT_ROOT):
    for file in files:
        if file.endswith(".json") and "possible_answer" not in root and ".cache" not in root:
            rel_path = os.path.relpath(os.path.join(root, file), INPUT_ROOT)
            all_json_files.append(rel_path)

print(f"[*] Ditemukan {len(all_json_files)} file JSON untuk diproses:\n")
for f in all_json_files:
    print(f"• {f}")

print("\n Memulai proses penerjemahan...\n")
for rel_path in all_json_files:
    src_file = os.path.join(INPUT_ROOT, rel_path)
    dst_file = os.path.join(OUTPUT_ROOT, rel_path)
    process_json_file(src_file, dst_file, translator_engine, batch_size=15)

print("\n SELURUH DATASET TELAGENTBENCH BERHASIL DITERJEMAHKAN KE BAHASA INDONESIA!")

## Section 5: Automated Quality Check & Residual Hangul Scanner

In [ ]:
print(" Menjalankan Quality Assurance & Residual Hangul Scanner...\n")

total_checked = 0
corrupted_files = []
residual_hangul = {}

for root, dirs, files in os.walk(OUTPUT_ROOT):
    for file in files:
        if file.endswith(".json"):
            total_checked += 1
            file_path = os.path.join(root, file)
            try:
                data, _ = load_universal_json(file_path)
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
                    hangul_matches = len(re.findall(r'[\uac00-\ud7af]', content))
                    if hangul_matches > 0:
                        residual_hangul[file] = hangul_matches
            except Exception as e:
                corrupted_files.append((file, str(e)))

print("==================================================")
print(f" LAPORAN QUALITY ASSURANCE ({total_checked} FILE JSON)")
print("==================================================")
if not corrupted_files:
    print(" 100% Format JSON Valid & Utuh (Tidak ada syntax error)!")
else:
    print(f" Ditemukan {len(corrupted_files)} file corrupt:")
    for cf, err in corrupted_files:
        print(f"• {cf}: {err}")

if not residual_hangul:
    print(" 100% Bersih dari karakter Korea (Hangul Residual = 0)!")
else:
    print(f" Sisa karakter Korea pada beberapa file:")
    for rf, count in residual_hangul.items():
        print(f"• {rf}: {count} karakter Hangul")
print("==================================================")

## Section 6: Export Archive

In [ ]:
zip_filename = "dataset_telagentbench_indonesia"
if os.path.exists("/kaggle/working"):
    zip_dest = "/kaggle/working/" + zip_filename
else:
    zip_dest = zip_filename

print(f" Mengompres folder '{OUTPUT_ROOT}' ke '{zip_dest}.zip'...")
shutil.make_archive(zip_dest, 'zip', OUTPUT_ROOT)

zip_size_mb = os.path.getsize(f"{zip_dest}.zip") / (1024 * 1024)
print(f" SELESAI! File siap diunduh di Kaggle Output: {zip_filename}.zip ({zip_size_mb:.2f} MB)")